# 2차 — Feature-Subspace Bagging

## 배경 (빠른 사전 테스트 결과)
| 피처 유지율 | 개별 모델 AUC | 모델간 상관관계 | 배깅 AUC(5개) |
|---|---|---|---|
| 75% | 0.715~0.738 (불안정) | 0.9188 | 0.73731 |
| 90% | 0.736~0.739 (안정적) | 0.9827 | 0.73977 |

90% 유지가 균형점으로 보임 — 지금까지 시도한 모든 모델(XGBoost/CatBoost/TabTransformer/
rank-loss/EBM, 전부 0.96~0.99+)보다 상관관계가 낮음. 모델 수를 5→10개로 늘리고, 실제
robust params로 정밀검증 + baseline과 블렌딩까지 확인.

## 메커니즘
멀티시드 배깅(같은 피처, 다른 seed)은 상관관계 0.99+로 디커럴레이션 효과가 거의 없었음.
Feature-subspace는 모델마다 **다른 피처를 통째로 못 보게** 만들어서 트리 구조 자체가
달라지므로, 같은 "배깅"이지만 진짜 다양성이 생김.

---
**저장 위치:** `experiment_history/2차/2차_feature_subspace_bagging.ipynb`


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
BLEND_DIR = SHARED_DIR / "blend_cache"

ROBUST_PARAMS_PATH = SHARED_DIR / "lgbm_robust_best_params.json"
FALLBACK_PARAMS_PATH = SHARED_DIR / "best_params.json"

N_SPLITS = 5
N_SUBSPACE_MODELS = 10  # 멀티시드 배깅과 동일한 개수로 비교 가능하게
FEATURE_FRACTION = 0.9  # 사전테스트에서 균형점이었던 값
SUBSPACE_SEED_BASE = 3000


## 1. 데이터 로드 + 전처리 (train + test 둘 다, 최종 test 예측까지 만들 것이므로)

In [3]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
test_raw = test_raw_full.drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
df_test = fill_na(base_features(test_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

# ★ 대회 규칙 준수: LabelEncoder는 train에만 fit
df_train_le, df_test_le = df_train.copy(), df_test.copy()
for col in cat_cols:
    le = LabelEncoder()
    train_vals = df_train_le[col].astype(str)
    le.fit(train_vals)
    known = set(le.classes_)
    fallback = train_vals.value_counts().idxmax()
    test_vals = df_test_le[col].astype(str)
    test_vals = test_vals.where(test_vals.isin(known), fallback)
    df_train_le[col] = le.transform(train_vals)
    df_test_le[col] = le.transform(test_vals)

y = df_train_le[TARGET_COL].values.astype(int)
X = df_train_le.drop(columns=[TARGET_COL])
X_test = df_test_le.copy()
all_cols = X.columns.tolist()
print(f"train {X.shape}, test {X_test.shape}, 전체 피처 {len(all_cols)}개")


train (256351, 67), test (90067, 67), 전체 피처 67개


## 2. 시작 파라미터 로드

In [4]:
def load_base_params(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        loaded = json.load(f)
    return loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

params_path = ROBUST_PARAMS_PATH if ROBUST_PARAMS_PATH.exists() else FALLBACK_PARAMS_PATH
base_params = load_base_params(params_path)
print(f"파라미터 출처: {params_path}")


파라미터 출처: ../lgbm_robust_best_params.json


## 3. Feature-subspace 배깅 (10개 모델, 90% 피처)

In [5]:
n_keep = int(len(all_cols) * FEATURE_FRACTION)
print(f"모델마다 {len(all_cols)}개 중 {n_keep}개 피처 무작위 사용")

oof_per_model = []
test_per_model = []
start = time.time()
for i in range(N_SUBSPACE_MODELS):
    seed = SUBSPACE_SEED_BASE + i
    rng_i = np.random.RandomState(seed)
    selected_cols = list(rng_i.choice(all_cols, size=n_keep, replace=False))
    X_sub, X_test_sub = X[selected_cols], X_test[selected_cols]

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test_sub))
    for tr_idx, val_idx in skf.split(X_sub, y):
        m = LGBMClassifier(**base_params, random_state=seed, verbose=-1)
        m.fit(X_sub.iloc[tr_idx], y[tr_idx])
        oof[val_idx] = m.predict_proba(X_sub.iloc[val_idx])[:, 1]
        test_pred += m.predict_proba(X_test_sub)[:, 1] / N_SPLITS

    oof_per_model.append(oof)
    test_per_model.append(test_pred)
    print(f"  모델 {i+1}/{N_SUBSPACE_MODELS} (seed={seed}): AUC={roc_auc_score(y, oof):.5f}  "
          f"({time.time()-start:.0f}s 누적)")

oof_subspace_bagged = np.mean(oof_per_model, axis=0)
test_subspace_bagged = np.mean(test_per_model, axis=0)

print(f"\nFeature-subspace 배깅({N_SUBSPACE_MODELS}개) OOF AUC: {roc_auc_score(y, oof_subspace_bagged):.5f}")

n = len(oof_per_model)
corrs = [np.corrcoef(oof_per_model[i], oof_per_model[j])[0, 1]
         for i in range(n) for j in range(i + 1, n)]
print(f"모델간 평균 pairwise 상관관계: {np.mean(corrs):.4f}")
print()
print("비교 기준:")
print("  10시드 배깅 baseline(전체 피처): 0.74036 / 0.74037")


모델마다 67개 중 60개 피처 무작위 사용
  모델 1/10 (seed=3000): AUC=0.72365  (16s 누적)
  모델 2/10 (seed=3001): AUC=0.73972  (33s 누적)
  모델 3/10 (seed=3002): AUC=0.73910  (50s 누적)
  모델 4/10 (seed=3003): AUC=0.73993  (67s 누적)
  모델 5/10 (seed=3004): AUC=0.72423  (83s 누적)
  모델 6/10 (seed=3005): AUC=0.73923  (99s 누적)
  모델 7/10 (seed=3006): AUC=0.73895  (116s 누적)
  모델 8/10 (seed=3007): AUC=0.73977  (133s 누적)
  모델 9/10 (seed=3008): AUC=0.73896  (149s 누적)
  모델 10/10 (seed=3009): AUC=0.73954  (166s 누적)

Feature-subspace 배깅(10개) OOF AUC: 0.73973
모델간 평균 pairwise 상관관계: 0.9724

비교 기준:
  10시드 배깅 baseline(전체 피처): 0.74036 / 0.74037


## 4. 기존 baseline과 블렌딩 (디커럴레이션이 실제로 도움 되는지)

In [6]:
LGBM_OOF_PATH = BLEND_DIR / "oof_10seed_bagged.npy"
LGBM_TEST_PATH = BLEND_DIR / "test_lgbm_bagged.npy"

if LGBM_OOF_PATH.exists():
    oof_baseline = np.load(LGBM_OOF_PATH)
    print(f"기존 baseline(10시드 배깅) OOF AUC: {roc_auc_score(y, oof_baseline):.5f}")
    print(f"Feature-subspace 배깅과의 상관관계: {np.corrcoef(oof_baseline, oof_subspace_bagged)[0,1]:.4f}")
    print(f"(참고: 멀티시드 배깅끼리는 보통 0.99+, 이게 그보다 낮으면 블렌딩 가치 있음)")
    print()

    best_score, best_w = roc_auc_score(y, oof_baseline), 1.0
    for w in np.arange(0, 1.01, 0.05):
        blend = w * oof_baseline + (1 - w) * oof_subspace_bagged
        score = roc_auc_score(y, blend)
        if score > best_score:
            best_score, best_w = score, w
    print(f"최적 블렌딩: baseline {best_w:.2f} + subspace {1-best_w:.2f}")
    print(f"블렌딩 OOF AUC: {best_score:.5f}  (baseline 단독 대비 {best_score - roc_auc_score(y, oof_baseline):+.5f})")
else:
    print(f"{LGBM_OOF_PATH} 없음 — 직접 경로 확인 후 비교해보세요")

BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "oof_feature_subspace.npy", oof_subspace_bagged)
np.save(BLEND_DIR / "test_feature_subspace.npy", test_subspace_bagged)
print("\n저장 완료: blend_cache/oof_feature_subspace.npy, test_feature_subspace.npy")


기존 baseline(10시드 배깅) OOF AUC: 0.74036
Feature-subspace 배깅과의 상관관계: 0.9970
(참고: 멀티시드 배깅끼리는 보통 0.99+, 이게 그보다 낮으면 블렌딩 가치 있음)

최적 블렌딩: baseline 0.95 + subspace 0.05
블렌딩 OOF AUC: 0.74036  (baseline 단독 대비 +0.00000)

저장 완료: blend_cache/oof_feature_subspace.npy, test_feature_subspace.npy


## 5. 해석 가이드

- 블렌딩 OOF AUC가 baseline 단독(0.74036~0.74037)보다 노이즈(0.00011)의 2~3배 이상 높으면 —
  진짜 개선. `oof_feature_subspace.npy`/`test_feature_subspace.npy`를 팀 블렌딩(v5 포함)에도
  추가해서 최종 제출 후보 점수가 0.74071을 넘는지 확인해볼 가치 있음
- 비슷하거나 낮으면 — feature-subspace도 상관관계는 낮췄지만 개별 성능 손실이 디커럴레이션
  이득을 상쇄했다는 뜻. `FEATURE_FRACTION`을 0.95 쪽으로 더 올려서 재시도해볼 수 있음
